# Landslide Dataset Audit v0.1

This notebook audits the GSI Landslide Inventory GeoJSON, filters for the 8 NER states, deduplicates records, parses the event year, generates negative samples, and prepares the base tabular dataset (`Landslide_Dataset_v0.1.csv`) for feature joining.

In [ ]:
import os
import json
import random
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Filter GSI GeoJSON

In [ ]:
base_path = "TERRAPULSE_DATA"
gsi_path = os.path.join(base_path, "01_LANDSLIDE_LABELS", "GSI_Landslide_Inventory.shp", "GSI_Landslide_Inventory.geojson")

print("Loading GSI GeoJSON...")
gdf = gpd.read_file(gsi_path)
print(f"Total raw records: {len(gdf)}")

# Clean state names
gdf['STATE'] = gdf['STATE'].astype(str).str.strip().str.title()

ner_states = [
    "Assam", "Arunachal Pradesh", "Meghalaya", "Mizoram", 
    "Nagaland", "Manipur", "Tripura", "Sikkim"
]

ner_gdf = gdf[gdf['STATE'].isin(ner_states)].copy()
print(f"NER records before deduplication: {len(ner_gdf)}")

## 2. Deduplicate and Extract Features

In [ ]:
# Deduplicate by SLIDE_NO (which acts as a unique ID for events)
ner_gdf = ner_gdf.drop_duplicates(subset=['SLIDE_NO']).copy()
print(f"NER records after deduplication: {len(ner_gdf)}")

# Extract coordinates
ner_gdf['LATITUDE'] = ner_gdf.geometry.y
ner_gdf['LONGITUDE'] = ner_gdf.geometry.x

# Parse Year from SLIDE_NO (Format: STATE_DISTRICT_YEAR_...)
def extract_year(slide_no):
    if pd.isna(slide_no): return None
    parts = str(slide_no).split('_')
    for p in parts:
        if len(p) == 4 and p.isdigit():
            year = int(p)
            if 1900 <= year <= 2025:
                return year
    return None

ner_gdf['EVENT_YEAR'] = ner_gdf['SLIDE_NO'].apply(extract_year)
print(f"Records with valid year: {ner_gdf['EVENT_YEAR'].notna().sum()}")

# Drop records without a valid year (needed for rainfall join later)
landslides = ner_gdf.dropna(subset=['EVENT_YEAR']).copy()
landslides['is_landslide'] = 1
print(f"Final positive landslide samples: {len(landslides)}")

## 3. Generate Negative Samples

In [ ]:
# Bounding box for NER
lat_min, lat_max = 21.5, 29.6
lon_min, lon_max = 87.5, 97.5

# We will generate a 1:1 ratio of negative samples
num_negatives = len(landslides)
neg_samples = []

for _ in range(num_negatives):
    neg_lat = random.uniform(lat_min, lat_max)
    neg_lon = random.uniform(lon_min, lon_max)
    # Random year between 2007 and 2022 to match the bulk of our data
    neg_year = random.randint(2007, 2022) 
    
    neg_samples.append({
        'SLIDE_NO': f"NEG_SAMPLE_{_}",
        'STATE': "UNKNOWN",
        'DISTRICT': "UNKNOWN",
        'LATITUDE': neg_lat,
        'LONGITUDE': neg_lon,
        'EVENT_YEAR': neg_year,
        'is_landslide': 0
    })

negatives_df = pd.DataFrame(neg_samples)
print(f"Generated negative samples: {len(negatives_df)}")

## 4. Combine and Save

In [ ]:
# Select relevant columns from positive samples
cols = ['SLIDE_NO', 'STATE', 'DISTRICT', 'LATITUDE', 'LONGITUDE', 'EVENT_YEAR', 'is_landslide']
positives_df = landslides[cols]

# Combine positives and negatives
final_dataset = pd.concat([positives_df, negatives_df], ignore_index=True)

# Shuffle the dataset
final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

out_dir = os.path.join(base_path, "99_RAW_UNVERIFIED")
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "Landslide_Dataset_v0.1.csv")

final_dataset.to_csv(out_file, index=False)
print(f"Saved final dataset ({len(final_dataset)} rows) to: {out_file}")
final_dataset.head()